In [7]:
from supabase import create_client
from dotenv import load_dotenv
import os, pandas as pd
import logging

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
df_orders = pd.DataFrame(supabase.table('sales').select('*').execute().data)
df_customers = pd.DataFrame(supabase.table('customers').select('*').execute().data)
print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")
print("--- TELLIMUSTE INFO ---")
df_orders.info()

print("\n--- KLIENDIDE INFO ---")
df_customers.info()

Tellimusi: 1000, Kliente: 1000
--- TELLIMUSTE INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sale_id         1000 non-null   int64  
 1   invoice_id      1000 non-null   str    
 2   sale_date       1000 non-null   str    
 3   customer_id     919 non-null    float64
 4   product_id      1000 non-null   int64  
 5   quantity        1000 non-null   int64  
 6   unit_price      1000 non-null   float64
 7   total_price     1000 non-null   float64
 8   channel         1000 non-null   str    
 9   store_location  687 non-null    str    
 10  payment_method  1000 non-null   str    
dtypes: float64(3), int64(3), str(5)
memory usage: 127.9 KB

--- KLIENDIDE INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -

In [9]:
# 1. Veendume, et logger on seadistatud
logger = logging.getLogger(__name__)
logger.info("Alustame Tartu linna tellimuste analüüsi...")

try:
    # 2. Ühendame baasandmed (Left join)
    df_merged = df_orders.merge(df_customers[['customer_id', 'city']], on='customer_id', how='left')
    
    # 3. Filtreerime välja ainult TARTU tellimused (koos tühikute ja suurtähtede kaitsega)
    # Kui tahad hoopis Pärnut, asenda 'tartu' lihtsalt sõnaga 'pärnu'
    df_tartu = df_merged[df_merged['city'].str.strip().str.lower() == 'tartu'].copy()
    
    if df_tartu.empty:
        logger.warning("Andmebaasis ei leitud Tartu linna kohta ühtegi tellimust!")
    else:
        # 4. ARVUTUSED: tellimuste arv, kogukäive ja keskmine tellimus
        tartu_count = len(df_tartu)
        tartu_revenue = df_tartu['total_price'].sum()
        tartu_average = df_tartu['total_price'].mean() # .mean() leiab keskmise
        
        # 5. Tulemuste väljastamine loggeri kaudu
        logger.info(f"Tartu analüüs lõpetatud edukalt.")
        logger.info(f">>> Tellimuste arv Tartus: {tartu_count} tk")
        logger.info(f">>> Kogukäive Tartus: {tartu_revenue:.2f} EUR")
        logger.info(f">>> Keskmine tellimuse summa Tartus: {tartu_average:.2f} EUR")
        
        # Sorteerime ja kuvame ka visuaalselt Tartu TOP 3 tellimused
        print("\nTOP 3 Tartu tellimust:")
        display(df_tartu.sort_values('total_price', ascending=False).head(3))

except KeyError as e:
    logger.error(f"Viga: kontrolli, kas veerud 'city' ja 'total_price' on tabelites olemas! Detailid: {e}")
except Exception as e:
    logger.critical(f"Ootamatu viga Tartu andmete analüüsimisel: {e}")

2026-05-20 18:42:28 - INFO - [2897069123.py:3] - Alustame Tartu linna tellimuste analüüsi...
2026-05-20 18:42:28 - INFO - [2897069123.py:22] - Tartu analüüs lõpetatud edukalt.
2026-05-20 18:42:28 - INFO - [2897069123.py:23] - >>> Tellimuste arv Tartus: 39 tk
2026-05-20 18:42:28 - INFO - [2897069123.py:24] - >>> Kogukäive Tartus: 11306.47 EUR
2026-05-20 18:42:28 - INFO - [2897069123.py:25] - >>> Keskmine tellimuse summa Tartus: 289.91 EUR



TOP 3 Tartu tellimust:


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,city
452,457,INV-202302-00196,2023-02-19,2318.0,1273,3,274.37,823.11,pood,Tallinn,järelmaks,Tartu
605,613,INV-202303-00074,2023-03-22,2375.0,1191,3,230.06,690.18,pood,Tallinn,sularaha,Tartu
657,666,INV-202303-00127,2023-03-14,2387.0,1296,2,301.97,603.94,pood,Tallinn,sularaha,Tartu


In [ ]:
logger = logging.getLogger(__name__)
logger.info("Alustan kõigi linnade ülese koondanalüüsi koostamist...")

try:
    # 1. Ühendame müügid ja kliendid
    df_merged = df_orders.merge(df_customers[['customer_id', 'city']], on='customer_id', how='left')
    
    # Puhastame linnade nimed igaks juhuks tühikutest ja teeme esisuurtäheks (nt "tallinn" -> "Tallinn")
    df_merged['city'] = df_merged['city'].str.strip().str.title()
    
    # 2. Grupeerime LINNA järgi ja arvutame igale linnale korraga kolm näitajat
    city_summary = df_merged.groupby('city').agg(
        tellimuste_arv=('total_price', 'count'),      # Mitu rida/tellimust
        kogukaive=('total_price', 'sum'),             # Summa kokku
        keskmine_tellimus=('total_price', 'mean')     # Keskmine tehing (.mean)
    ).reset_index()
    
    # Sorteerime kogukäibe järgi, et kõige suurem linn oleks eespool
    city_summary = city_summary.sort_values('kogukaive', ascending=False)
    
    logger.info("Kõigi linnade koondtabel on edukalt arvutatud!")
    
    # Kuvame ilusa koondtabeli
    print("\n LINNADE VÕRDLUSTABEL:")
    display(city_summary)

except Exception as e:
    logger.error(f"Viga linnade koondtabeli arvutamisel: {e}")

2026-05-20 18:43:59 - INFO - [2287493925.py:4] - Alustan kõigi linnade ülese koondanalüüsi koostamist...
2026-05-20 18:43:59 - INFO - [2287493925.py:23] - Kõigi linnade koondtabel on edukalt arvutatud!



 LINNADE VÕRDLUSTABEL:


,city,tellimuste_arv,kogukaive,keskmine_tellimus
7,Tallinn,116,33526.46,289.021207
5,Pärnu,39,12638.36,324.060513
8,Tartu,39,11306.47,289.909487
2,Kuressaare,12,5677.86,473.155000
10,Viljandi,9,3624.94,402.771111
1,Jõhvi,10,3475.31,347.531000
9,Valga,9,2580.36,286.706667
3,Narva,15,2358.98,157.265333
0,Haapsalu,9,1967.47,218.607778
6,Rakvere,6,1728.25,288.041667


In [11]:
logger = logging.getLogger(__name__)
logger.info("Alustan suurte tellimuste (kliendivaalade) analüüsi...")

try:
    # ÄRIKÜSIMUS: Kes teevad meil tellimusi väärtusega üle 100€ ja mis on nende keskmine ost?
    # 1. Filtreerime välja ainult tellimused, mille hind on suurem kui 100€
    df_large_orders = df_orders[df_orders['total_price'] > 100].copy()
    
    # 2. Kontrollime tulemusi (Vastavalt kontrolltabelile: shape, head, describe)
    logger.info("Andmete filtreerimine edukas. Alustan tulemuste kontrolli.")
    
    print("\n=== 1. ANDMETABELI KUJU (shape) ===")
    print(f"Suuri tellimusi leiti: {df_large_orders.shape[0]} tükki (veerge: {df_large_orders.shape[1]})")
    
    print("\n=== 2. ESIMESED READ (head) ===")
    display(df_large_orders.head(3))
    
    print("\n=== 3. STATISTILINE KOKKUVÕTE (describe) ===")
    display(df_large_orders[['total_price']].describe())

except Exception as e:
    logger.error(f"Päringu käivitamisel tekkis viga: {e}")

2026-05-20 18:47:59 - INFO - [3158332567.py:2] - Alustan suurte tellimuste (kliendivaalade) analüüsi...
2026-05-20 18:47:59 - INFO - [3158332567.py:10] - Andmete filtreerimine edukas. Alustan tulemuste kontrolli.



=== 1. ANDMETABELI KUJU (shape) ===
Suuri tellimusi leiti: 795 tükki (veerge: 11)

=== 2. ESIMESED READ (head) ===


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks



=== 3. STATISTILINE KOKKUVÕTE (describe) ===


,total_price
count,795.000000
mean,349.910830
std,261.213931
min,100.400000
25%,176.650000
50%,264.600000
75%,436.670000
max,1792.050000
